In [181]:
%pip install -q nfl-data-py

import pandas as pd
import numpy as np
import nfl_data_py as nfl


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [182]:
from datetime import datetime

# Set the season to analyze (e.g., 2024). Change if needed.
season = 2025
week = 1

# Load play-by-play data for the chosen season
pbp = nfl.import_pbp_data(years=[2025])

# Filter for regular season, Week 1
week_pbp = pbp[(pbp['week'] == week)]

# Keep only touchdown plays
week_tds = week_pbp[week_pbp['touchdown'] == 1]

# Count TDs per scorer. Prefer id+name if both available, else fall back to name only
use_cols = [c for c in ['td_player_id', 'td_player_name'] if c in week_tds.columns]
if use_cols:
    scorers = (
        week_tds.dropna(subset=use_cols)
        .groupby(use_cols)
        .size()
        .reset_index(name='tds')
    )
    if 'td_player_id' in use_cols:
        scorers = scorers.rename(columns={'td_player_name': 'player', 'td_player_id': 'player_id'})
    else:
        scorers = scorers.rename(columns={'td_player_name': 'player'})
else:
    # Fallback if td_* columns not present; derive from rusher/receiver
    rush = week_tds.dropna(subset=['rusher_player_id'])[['rusher_player_id', 'rusher_player_name']]
    rec = week_tds.dropna(subset=['receiver_player_id'])[['receiver_player_id', 'receiver_player_name']]
    rush.columns = ['player_id', 'player']
    rec.columns = ['player_id', 'player']
    both = pd.concat([rush, rec], ignore_index=True)
    scorers = both.groupby(['player_id', 'player']).size().reset_index(name='tds')

# Show results
scorers.head(50)


2025 done.
Downcasting floats.


,player_id,player,tds
0,00-0030061,Z.Ertz,1
1,00-0030279,K.Allen,1
2,00-0030506,T.Kelce,1
3,00-0030564,D.Hopkins,1
4,00-0032764,D.Henry,2
5,00-0033288,G.Kittle,1
6,00-0033293,A.Jones,1
7,00-0033553,J.Conner,1
8,00-0033858,J.Smith,1
9,00-0033873,P.Mahomes,1


In [183]:
predictions = pd.read_csv(f'data/predictions_week_{week}.csv')

#Keep only player_id, player_display_name, predicted_touchdown_probability, and model_edge
predictions = predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge','market_implied_prob']]

#Sort by predicted_touchdown_probability in descending order
predictions = predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

#Show the top 50 players
predictions.head(50)



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0032764,Derrick Henry,RB,BAL,0.610808,-145.0,0.018972,0.591837
1,00-0039361,Bucky Irving,RB,TB,0.597314,-140.0,0.013981,0.583333
2,00-0039040,De'Von Achane,RB,MIA,0.574486,-140.0,-0.008847,0.583333
3,00-0034844,Saquon Barkley,RB,PHI,0.573298,-185.0,-0.075825,0.649123
4,00-0035700,Josh Jacobs,RB,GB,0.570906,-160.0,-0.044478,0.615385
5,00-0033553,James Conner,RB,ARI,0.570442,-155.0,-0.037401,0.607843
6,00-0036555,Chuba Hubbard,RB,CAR,0.548293,-105.0,0.036098,0.512195
7,00-0037840,Kyren Williams,RB,LA,0.548109,-140.0,-0.035224,0.583333
8,00-0039139,Jahmyr Gibbs,RB,DET,0.547903,-105.0,0.035707,0.512195
9,00-0038597,Chase Brown,RB,CIN,0.547071,-150.0,-0.052929,0.600000


In [184]:
# Join predictions with scorers: prefer player_id, else fall back to name
if 'player_id' in scorers.columns:
    pred_scored = predictions.merge(
        scorers[['player_id', 'tds']], on='player_id', how='inner'
    )
else:
    pred_scored = predictions.merge(
        scorers[['player', 'tds']], left_on='player_display_name', right_on='player', how='inner'
    )

pred_scored.sort_values(['predicted_touchdown_probability'], ascending=[False])

,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob,tds
0,00-0032764,Derrick Henry,RB,BAL,0.610808,-145.0,0.018972,0.591837,2
1,00-0039361,Bucky Irving,RB,TB,0.597314,-140.0,0.013981,0.583333,1
2,00-0039040,De'Von Achane,RB,MIA,0.574486,-140.0,-0.008847,0.583333,1
3,00-0034844,Saquon Barkley,RB,PHI,0.573298,-185.0,-0.075825,0.649123,1
4,00-0035700,Josh Jacobs,RB,GB,0.570906,-160.0,-0.044478,0.615385,1
5,00-0033553,James Conner,RB,ARI,0.570442,-155.0,-0.037401,0.607843,1
6,00-0036555,Chuba Hubbard,RB,CAR,0.548293,-105.0,0.036098,0.512195,1
7,00-0037840,Kyren Williams,RB,LA,0.548109,-140.0,-0.035224,0.583333,1
8,00-0038597,Chase Brown,RB,CIN,0.547071,-150.0,-0.052929,0.600000,1
9,00-0038542,Bijan Robinson,RB,ATL,0.545828,-175.0,-0.090536,0.636364,1


In [185]:
###Simulate betting on the top 10 running backs
def simulate_betting(df, scorers):
    stake = 10.0

    bets = df.copy()

    # Merge to mark hits
    bets = bets.merge(
        scorers[['player_id', 'tds']], on='player_id', how='left'
    )
    bets['tds'] = bets['tds'].fillna(0).astype(int)
    bets['hit'] = bets['tds'] > 0

    # American odds payout logic
    # profit_if_win = stake * (odds/100) if odds > 0 else stake * (100/abs(odds))
    # profit_if_loss = -stake
    is_plus = bets['price'] > 0
    profit_if_win = stake * (bets['price'] / 100.0)
    profit_if_win = profit_if_win.where(is_plus, stake * (100.0 / bets['price'].abs()))

    bets['profit'] = np.where(bets['hit'], profit_if_win, -stake)

    # Add total return
    bets['return'] = stake + bets['profit']

    # Summary metrics
    num_bets = len(bets)
    hits = int(bets['hit'].sum())
    hit_rate = hits / num_bets if num_bets else 0.0
    total_profit = float(bets['profit'].sum())
    roi = total_profit / (stake * num_bets) if num_bets else 0.0

    summary = {
        'bets': num_bets,
        'hits': hits,
        'hit_rate': round(hit_rate, 3),
        'total_profit': round(total_profit, 2),
        'roi': round(roi, 3)
    }

    display(summary)

    # Show detailed results
    cols = [
        'player_id', 'player_display_name', 'team', 'position', 'price',
        'predicted_touchdown_probability', 'model_edge', 'tds', 'hit', 'profit', 'return'
    ]
    return bets[cols].sort_values(['predicted_touchdown_probability'], ascending=[False]).reset_index(drop=True)



In [186]:
# output players from predictions that play for 'PHI', 'KC', 'LAC' or 'DAL'

#find players with model_edge > 0 and price < 500
ev = predictions[predictions['model_edge'] > 0.10] 
ev = ev[ev['price'] < 500]

#sort by model_edge in descending order
ev = ev.sort_values('model_edge', ascending=False)
ev







,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
27,00-0038544,Quentin Johnston,WR,LAC,0.466127,370.0,0.253361,0.212766
36,00-0033858,Jonnu Smith,TE,PIT,0.434329,400.0,0.234329,0.200000
46,00-0036894,Pat Freiermuth,TE,PIT,0.405808,425.0,0.215332,0.190476
48,00-0038117,Wan'Dale Robinson,WR,NYG,0.403713,400.0,0.203713,0.200000
38,00-0036139,Rico Dowdle,RB,CAR,0.430344,320.0,0.192249,0.238095
12,00-0036912,DeVonta Smith,WR,PHI,0.539514,180.0,0.182371,0.357143
58,00-0039165,Zach Charbonnet,RB,SEA,0.385771,370.0,0.173005,0.212766
26,00-0034960,Jakobi Meyers,WR,LV,0.466541,225.0,0.158849,0.307692
21,00-0037744,Trey McBride,TE,ARI,0.482534,200.0,0.149201,0.333333
35,00-0033885,David Njoku,TE,CLE,0.436788,230.0,0.133757,0.303030


In [187]:
simulate_betting(ev, scorers)

{'bets': 19, 'hits': 6, 'hit_rate': 0.316, 'total_profit': 38.0, 'roi': 0.2}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036912,DeVonta Smith,PHI,WR,180.0,0.539514,0.182371,0,False,-10.0,0.0
1,00-0036158,J.K. Dobbins,DEN,RB,160.0,0.514653,0.130037,1,True,16.0,26.0
2,00-0035676,A.J. Brown,PHI,WR,160.0,0.503039,0.118423,0,False,-10.0,0.0
3,00-0033293,Aaron Jones,MIN,RB,165.0,0.482696,0.105338,1,True,16.5,26.5
4,00-0037744,Trey McBride,ARI,TE,200.0,0.482534,0.149201,0,False,-10.0,0.0
5,00-0034960,Jakobi Meyers,LV,WR,225.0,0.466541,0.158849,0,False,-10.0,0.0
6,00-0038544,Quentin Johnston,LAC,WR,370.0,0.466127,0.253361,2,True,37.0,47.0
7,00-0034827,DJ Moore,CHI,WR,195.0,0.465005,0.126022,0,False,-10.0,0.0
8,00-0036252,Michael Pittman,IND,WR,215.0,0.446195,0.128735,1,True,21.5,31.5
9,00-0033885,David Njoku,CLE,TE,230.0,0.436788,0.133757,0,False,-10.0,0.0


In [242]:
### Get top 10 rb, wr, te, qb from predictions
top_rb = predictions[predictions['position'] == 'RB'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_wr = predictions[predictions['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(15)
top_te = predictions[predictions['position'] == 'TE'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
top_qb = predictions[predictions['position'] == 'QB'].sort_values('predicted_touchdown_probability', ascending=False).head(5)

top_rb



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0032764,Derrick Henry,RB,BAL,0.610808,-145.0,0.018972,0.591837
1,00-0039361,Bucky Irving,RB,TB,0.597314,-140.0,0.013981,0.583333
2,00-0039040,De'Von Achane,RB,MIA,0.574486,-140.0,-0.008847,0.583333
3,00-0034844,Saquon Barkley,RB,PHI,0.573298,-185.0,-0.075825,0.649123
4,00-0035700,Josh Jacobs,RB,GB,0.570906,-160.0,-0.044478,0.615385
5,00-0033553,James Conner,RB,ARI,0.570442,-155.0,-0.037401,0.607843
6,00-0036555,Chuba Hubbard,RB,CAR,0.548293,-105.0,0.036098,0.512195
7,00-0037840,Kyren Williams,RB,LA,0.548109,-140.0,-0.035224,0.583333
8,00-0039139,Jahmyr Gibbs,RB,DET,0.547903,-105.0,0.035707,0.512195
9,00-0038597,Chase Brown,RB,CIN,0.547071,-150.0,-0.052929,0.600000


In [243]:
simulate_betting(top_rb, scorers)


{'bets': 10, 'hits': 9, 'hit_rate': 0.9, 'total_profit': 52.62, 'roi': 0.526}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.610808,0.018972,2,True,6.896552,16.896552
1,00-0039361,Bucky Irving,TB,RB,-140.0,0.597314,0.013981,1,True,7.142857,17.142857
2,00-0039040,De'Von Achane,MIA,RB,-140.0,0.574486,-0.008847,1,True,7.142857,17.142857
3,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.573298,-0.075825,1,True,5.405405,15.405405
4,00-0035700,Josh Jacobs,GB,RB,-160.0,0.570906,-0.044478,1,True,6.250000,16.250000
5,00-0033553,James Conner,ARI,RB,-155.0,0.570442,-0.037401,1,True,6.451613,16.451613
6,00-0036555,Chuba Hubbard,CAR,RB,-105.0,0.548293,0.036098,1,True,9.523810,19.523810
7,00-0037840,Kyren Williams,LA,RB,-140.0,0.548109,-0.035224,1,True,7.142857,17.142857
8,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.547903,0.035707,0,False,-10.000000,0.000000
9,00-0038597,Chase Brown,CIN,RB,-150.0,0.547071,-0.052929,1,True,6.666667,16.666667


In [244]:
#filter top_wr to only include players with model_edge > 0.05 and price < 500
#top_wr = top_wr[top_wr['model_edge'] > 0.05]
#top_wr = top_wr[top_wr['price'] < 500]
simulate_betting(top_wr, scorers)


{'bets': 15,
 'hits': 4,
 'hit_rate': 0.267,
 'total_profit': -25.0,
 'roi': -0.167}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036912,DeVonta Smith,PHI,WR,180.0,0.539514,0.182371,0,False,-10.0,0.0
1,00-0036900,Ja'Marr Chase,CIN,WR,-130.0,0.536732,-0.028486,0,False,-10.0,0.0
2,00-0039893,Brian Thomas Jr.,JAX,WR,130.0,0.518793,0.084011,1,True,13.0,23.0
3,00-0031408,Mike Evans,TB,WR,110.0,0.515207,0.039016,0,False,-10.0,0.0
4,00-0035676,A.J. Brown,PHI,WR,160.0,0.503039,0.118423,0,False,-10.0,0.0
5,00-0035659,Terry McLaurin,WAS,WR,130.0,0.503017,0.068234,0,False,-10.0,0.0
6,00-0039075,Puka Nacua,LA,WR,140.0,0.481319,0.064652,0,False,-10.0,0.0
7,00-0034348,Courtland Sutton,DEN,WR,135.0,0.478425,0.052893,1,True,13.5,23.5
8,00-0034960,Jakobi Meyers,LV,WR,225.0,0.466541,0.158849,0,False,-10.0,0.0
9,00-0038544,Quentin Johnston,LAC,WR,370.0,0.466127,0.253361,2,True,37.0,47.0


In [245]:
simulate_betting(top_te, scorers)

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 26.5, 'roi': 0.53}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0037744,Trey McBride,ARI,TE,200.0,0.482534,0.149201,0,False,-10.0,0.0
1,00-0030506,Travis Kelce,KC,TE,165.0,0.472411,0.095053,1,True,16.5,26.5
2,00-0033885,David Njoku,CLE,TE,230.0,0.436788,0.133757,0,False,-10.0,0.0
3,00-0033858,Jonnu Smith,PIT,TE,400.0,0.434329,0.234329,1,True,40.0,50.0
4,00-0034753,Mark Andrews,BAL,TE,205.0,0.421154,0.093285,0,False,-10.0,0.0


In [246]:
simulate_betting(top_qb, scorers)

{'bets': 5, 'hits': 4, 'hit_rate': 0.8, 'total_profit': 44.5, 'roi': 0.89}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0034857,Josh Allen,BUF,QB,-120.0,0.471089,-0.074365,2,True,8.333333,18.333333
1,00-0036389,Jalen Hurts,PHI,QB,-150.0,0.441697,-0.158303,2,True,6.666667,16.666667
2,00-0034796,Lamar Jackson,BAL,QB,205.0,0.387036,0.059168,1,True,20.500000,30.500000
3,00-0035710,Daniel Jones,IND,QB,190.0,0.381421,0.036594,2,True,19.000000,29.000000
4,00-0039910,Jayden Daniels,WAS,QB,170.0,0.261250,-0.109120,0,False,-10.000000,0.000000


In [247]:
top_predictors = predictions[predictions['predicted_touchdown_probability'] >= 0.50]
simulate_betting(top_predictors, scorers)

{'bets': 20, 'hits': 13, 'hit_rate': 0.65, 'total_profit': 37.84, 'roi': 0.189}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.610808,0.018972,2,True,6.896552,16.896552
1,00-0039361,Bucky Irving,TB,RB,-140.0,0.597314,0.013981,1,True,7.142857,17.142857
2,00-0039040,De'Von Achane,MIA,RB,-140.0,0.574486,-0.008847,1,True,7.142857,17.142857
3,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.573298,-0.075825,1,True,5.405405,15.405405
4,00-0035700,Josh Jacobs,GB,RB,-160.0,0.570906,-0.044478,1,True,6.250000,16.250000
5,00-0033553,James Conner,ARI,RB,-155.0,0.570442,-0.037401,1,True,6.451613,16.451613
6,00-0036555,Chuba Hubbard,CAR,RB,-105.0,0.548293,0.036098,1,True,9.523810,19.523810
7,00-0037840,Kyren Williams,LA,RB,-140.0,0.548109,-0.035224,1,True,7.142857,17.142857
8,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.547903,0.035707,0,False,-10.000000,0.000000
9,00-0038597,Chase Brown,CIN,RB,-150.0,0.547071,-0.052929,1,True,6.666667,16.666667


In [248]:
vegas_similar = predictions[predictions['model_edge'] < 0.05]
vegas_similar = vegas_similar[vegas_similar['model_edge'] > 0]
vegas_similar = vegas_similar[vegas_similar['price'] < 300]
simulate_betting(vegas_similar, scorers)

{'bets': 23,
 'hits': 9,
 'hit_rate': 0.391,
 'total_profit': -2.44,
 'roi': -0.011}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.610808,0.018972,2,True,6.896552,16.896552
1,00-0039361,Bucky Irving,TB,RB,-140.0,0.597314,0.013981,1,True,7.142857,17.142857
2,00-0036555,Chuba Hubbard,CAR,RB,-105.0,0.548293,0.036098,1,True,9.523810,19.523810
3,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.547903,0.035707,0,False,-10.000000,0.000000
4,00-0037248,James Cook,BUF,RB,105.0,0.528006,0.040201,1,True,10.500000,20.500000
5,00-0031408,Mike Evans,TB,WR,110.0,0.515207,0.039016,0,False,-10.000000,0.000000
6,00-0037238,Drake London,ATL,WR,125.0,0.455290,0.010846,0,False,-10.000000,0.000000
7,00-0039849,Marvin Harrison Jr.,ARI,WR,150.0,0.403806,0.003806,1,True,15.000000,25.000000
8,00-0039384,Tyrone Tracy Jr.,NYG,RB,160.0,0.400951,0.016336,0,False,-10.000000,0.000000
9,00-0036893,Najee Harris,LAC,RB,185.0,0.386058,0.035180,0,False,-10.000000,0.000000


In [249]:
top_25 = predictions.head(25)
simulate_betting(top_25, scorers)

{'bets': 25, 'hits': 16, 'hit_rate': 0.64, 'total_profit': 64.34, 'roi': 0.257}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0032764,Derrick Henry,BAL,RB,-145.0,0.610808,0.018972,2,True,6.896552,16.896552
1,00-0039361,Bucky Irving,TB,RB,-140.0,0.597314,0.013981,1,True,7.142857,17.142857
2,00-0039040,De'Von Achane,MIA,RB,-140.0,0.574486,-0.008847,1,True,7.142857,17.142857
3,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.573298,-0.075825,1,True,5.405405,15.405405
4,00-0035700,Josh Jacobs,GB,RB,-160.0,0.570906,-0.044478,1,True,6.250000,16.250000
5,00-0033553,James Conner,ARI,RB,-155.0,0.570442,-0.037401,1,True,6.451613,16.451613
6,00-0036555,Chuba Hubbard,CAR,RB,-105.0,0.548293,0.036098,1,True,9.523810,19.523810
7,00-0037840,Kyren Williams,LA,RB,-140.0,0.548109,-0.035224,1,True,7.142857,17.142857
8,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.547903,0.035707,0,False,-10.000000,0.000000
9,00-0038597,Chase Brown,CIN,RB,-150.0,0.547071,-0.052929,1,True,6.666667,16.666667
